In [1]:
# Cell 1: Install and imports
!conda install -y -q xgboost 2>/dev/null || pip install -q --prefer-binary xgboost

import os
import io
import json
import tarfile
import tempfile
import boto3
import joblib
import numpy as np
import pandas as pd

from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from xgboost import XGBClassifier

import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/python3

  added / updated specs:
    - xgboost


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    libxgboost-3.2.0           |   cpu_h2ebb00f_1         3.8 MB  conda-forge
    numpy-2.4.3                |  py312h33ff503_0         8.4 MB  conda-forge
    openssl-3.6.2              |       h35e630c_0         3.0 MB  conda-forge
    py-xgboost-3.2.0           | cpu_pyh718b53a_1         168 KB  conda-forge
    xgboost-3.2.0              | cpu_pyhb39878e_1          15 KB  conda-forge
    ------------------------------------------------------------
                                           Total:        15.5 MB

The follo

In [2]:
# Cell 2: Config
REGION = boto3.Session().region_name
BUCKET = "fraud-ml-25249347-eu-north-1"
KEY = "raw/creditcard.csv"
MODEL_PREFIX = "models/fraud-xgb"
ENDPOINT_NAME = "fraud-xgb-endpoint-" + datetime.now().strftime("%Y%m%d%H%M%S")

s3 = boto3.client("s3", region_name=REGION)
sm_session = sagemaker.Session(default_bucket=BUCKET)
role = get_execution_role()

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Endpoint name:", ENDPOINT_NAME)

Region: eu-north-1
Bucket: fraud-ml-25249347-eu-north-1
Endpoint name: fraud-xgb-endpoint-20260422185332


In [3]:
# Cell 3: Load data from S3
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print("Shape:", df.shape)
print(df.head(3))
print(df["Class"].value_counts())

Shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   

        V26       V27       V28  Amount  Class  
0 -0.189115  0.133558 -0.021053  149.62      0  
1  0.125895 -0.008983  0.014724    2.69      0  
2 -0.139097 -0.055353 -0.059752  378.66      0  

[3 rows x 31 columns]
Class
0    284315
1       492
Name: count, dtype: int64


In [4]:
# Cell 4: Prepare train/test
target_col = "Class"
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = float(neg) / float(pos)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("scale_pos_weight:", scale_pos_weight)

Train shape: (227845, 30)
Test shape: (56962, 30)
scale_pos_weight: 577.2868020304569


In [5]:
# Cell 5: Train XGBoost
model = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

print("Training complete!")
print("Precision:", precision_score(y_test, (probs >= 0.5).astype(int)))
print("Recall:", recall_score(y_test, (probs >= 0.5).astype(int)))
print("F1:", f1_score(y_test, (probs >= 0.5).astype(int)))
print("ROC-AUC:", roc_auc_score(y_test, probs))
print("Avg Precision:", average_precision_score(y_test, probs))

Training complete!
Precision: 0.8541666666666666
Recall: 0.8367346938775511
F1: 0.845360824742268
ROC-AUC: 0.977447443524399
Avg Precision: 0.8777733995263268


In [6]:
# Cell 6: Threshold tuning + metrics
best_t = 0.5
best_f1 = -1.0

for t in np.arange(0.05, 0.95, 0.01):
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_t = float(t)

final_preds = (probs >= best_t).astype(int)

metrics = {
    "threshold": best_t,
    "precision": precision_score(y_test, final_preds, zero_division=0),
    "recall": recall_score(y_test, final_preds, zero_division=0),
    "f1": f1_score(y_test, final_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test, probs),
    "pr_auc": average_precision_score(y_test, probs)
}

print("Metrics:", metrics)

Metrics: {'threshold': 0.8700000000000002, 'precision': 0.9302325581395349, 'recall': 0.8163265306122449, 'f1': 0.8695652173913043, 'roc_auc': 0.977447443524399, 'pr_auc': 0.8777733995263268}


In [7]:
import json
import os
import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# Make sure these already exist from previous cells:
# model, metrics, feature_cols, X_test, y_test, final_preds

os.makedirs("artifacts", exist_ok=True)

# Save model and metadata
joblib.dump(model, "artifacts/fraud_xgb_model.joblib")

with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open("artifacts/feature_cols.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

# Save evaluation outputs
cm = confusion_matrix(y_test, final_preds)
pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"]).to_csv("artifacts/confusion_matrix.csv")

report = classification_report(y_test, final_preds, output_dict=True, zero_division=0)
pd.DataFrame(report).transpose().to_csv("artifacts/classification_report.csv")

print("Saved model, metrics, feature list, confusion matrix, and classification report in artifacts/")
print("Best threshold:", metrics["threshold"])
print("Precision:", metrics["precision"])
print("Recall:", metrics["recall"])
print("F1:", metrics["f1"])
print("ROC-AUC:", metrics["roc_auc"])
print("PR-AUC:", metrics["pr_auc"])

Saved model, metrics, feature list, confusion matrix, and classification report in artifacts/
Best threshold: 0.8700000000000002
Precision: 0.9302325581395349
Recall: 0.8163265306122449
F1: 0.8695652173913043
ROC-AUC: 0.977447443524399
PR-AUC: 0.8777733995263268


In [9]:
import os
import json
import sagemaker
import boto3
from sagemaker import get_execution_role
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

In [10]:
# Create the serving script in the notebook working directory
inference_code = '''
import io
import os
import json
import joblib
import pandas as pd

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    with open(os.path.join(model_dir, "meta.json"), "r") as f:
        meta = json.load(f)
    return {"model": model, "meta": meta}

def input_fn(request_body, request_content_type):
    if request_content_type == "text/csv":
        return pd.read_csv(io.StringIO(request_body), header=None)
    raise ValueError("Unsupported content type")

def predict_fn(input_data, model_artifacts):
    model = model_artifacts["model"]
    threshold = float(model_artifacts["meta"]["threshold"])
    p = model.predict_proba(input_data)[:, 1][0]
    y = int(p >= threshold)
    return {"fraud_probability": float(p), "predicted_label": y, "threshold_used": threshold}

def output_fn(prediction, accept):
    return json.dumps(prediction), "application/json"
'''

with open("inference.py", "w") as f:
    f.write(inference_code)

import os
print("Working dir:", os.getcwd())
print("Script created:", os.path.exists("inference.py"))

Working dir: /home/ec2-user/SageMaker
Script created: True


In [14]:
# Direct inference test (no endpoint needed)
sample_features = X_test.iloc[0:5]  # 5 test samples

# Load saved model
import joblib
with open("artifacts/metrics.json", "r") as f:
    saved_metrics = json.load(f)

model = joblib.load("artifacts/fraud_xgb_model.joblib")
threshold = saved_metrics["threshold"]

# Make predictions
probs = model.predict_proba(sample_features)[:, 1]
preds = (probs >= threshold).astype(int)

# Show results
for i, (prob, pred) in enumerate(zip(probs, preds)):
    result = {
        "sample": i,
        "fraud_probability": float(prob),
        "predicted_label": int(pred),
        "threshold_used": threshold
    }
    print(result)

print("\nModel working - ready for Lambda integration")

{'sample': 0, 'fraud_probability': 1.1017498309229268e-06, 'predicted_label': 0, 'threshold_used': 0.8700000000000002}
{'sample': 1, 'fraud_probability': 3.1921435947879218e-06, 'predicted_label': 0, 'threshold_used': 0.8700000000000002}
{'sample': 2, 'fraud_probability': 1.040155257214792e-05, 'predicted_label': 0, 'threshold_used': 0.8700000000000002}
{'sample': 3, 'fraud_probability': 1.093459559342591e-06, 'predicted_label': 0, 'threshold_used': 0.8700000000000002}
{'sample': 4, 'fraud_probability': 0.0001824384235078469, 'predicted_label': 0, 'threshold_used': 0.8700000000000002}

Model working - ready for Lambda integration


In [ ]:
import os
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

# 1) Create a clean source folder with only inference.py
code_dir = "/tmp/sm_code"
os.makedirs(code_dir, exist_ok=True)

with open(f"{code_dir}/inference.py", "w") as f:
    f.write(inference_code)

print("inference exists:", os.path.exists(f"{code_dir}/inference.py"))

# 2) Reuse your already uploaded model artifact
model_data = "s3://fraud-ml-25249347-eu-north-1/models/fraud-xgb/model.tar.gz"

# 3) Deploy from clean source dir
sk_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point="inference.py",
    source_dir=code_dir,
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sm_session
)

predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    wait=True
)

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

print("Endpoint in service:", ENDPOINT_NAME)

In [20]:
import time
import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name="eu-north-1")
endpoint_name = "fraud-xgb-endpoint-20260422185436"

# 1) Wait until endpoint leaves in-progress states
while True:
    desc = sm.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Status:", status)
    if status in ["InService", "Failed", "OutOfService"]:
        break
    time.sleep(30)

# 2) Delete endpoint safely
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print("Delete requested for endpoint.")
except ClientError as e:
    print("Delete endpoint error:", e)

# 3) Wait a bit, then delete endpoint config (optional cleanup)
time.sleep(10)
try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print("Deleted endpoint config.")
except ClientError as e:
    print("Delete endpoint config error (ok if still in use):", e)

Status: Creating
Status: Failed
Delete requested for endpoint.
Deleted endpoint config.
